# Comparing RAG Methods

The goal of this project is to test different types of Retrieval Augmented Generation methods.

### Imports

In [1]:
# Core LangChain runtime
%pip install langchain-core
%pip install langchain-community langchain-text-splitters
%pip install sentence-transformers datasets

# Data
%pip install beir

# Vector store backends
%pip install faiss-cpu chromadb qdrant-client langchain-qdrant pywin32

# General utilities
%pip install numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.vectorstores import FAISS, Chroma, Pinecone
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore

from beir.datasets.data_loader import GenericDataLoader

import numpy as np
import random

c:\Users\sam_m\Documents\RAG-Comparison\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Generate Corpus

In [3]:
from beir import util

dataset = "nfcorpus"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
data_folder = f"./datasets/{dataset}"

# Downloads and unzips automatically
corpus_path = util.download_and_unzip(url, data_folder)

In [4]:
dataset = "nfcorpus"  # can replace with "msmarco", "trec-covid", etc.
data_folder = f"datasets/{dataset}/{dataset}"

corpus, queries, qrels = GenericDataLoader(data_folder).load(split="test")

100%|██████████| 3633/3633 [00:00<00:00, 150147.87it/s]


In [5]:
documents = [{"text": d["text"], "metadata": {"id": k}} for k, d in corpus.items()]
texts = [d["text"] for d in documents]
metadatas = [d["metadata"] for d in documents]

In [6]:
documents[0:3]

[{'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants died, of which 3,619 (60.2%) was due to breast cancer. After adjustment for age, tumor 

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = []
for doc in documents:
    for chunk_text in text_splitter.split_text(doc["text"]):
        chunks.append({"id": doc["metadata"]["id"], "text": chunk_text})

### 2. Create Retrievers

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\sam_m\AppData\Local\Temp\ipykernel_25560\2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


In [9]:
def build_vectorstore(store_type="faiss"):
    if store_type == "faiss":
        store = FAISS.from_texts([c["text"] for c in chunks],
                                 embedding_model,
                                 metadatas=[{"id": c["id"]} for c in chunks])
    elif store_type == "chroma":
        store = Chroma.from_texts([c["text"] for c in chunks],
                                  embedding_model,
                                  metadatas=[{"id": c["id"]} for c in chunks])
    elif store_type == "qdrant":
        client = QdrantClient(path="./qdrant_data")
        dim = len(embedding_model.embed_query("x"))
        if "ragtest" not in [i.name for i in client.get_collections().collections]:
            print("COLLECTIONS", client.get_collections().collections)
            client.create_collection(
                collection_name="ragtest",
                vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
            )
        store = QdrantVectorStore(
            client=client,
            collection_name="ragtest",
            embedding=embedding_model,
        )
        store.add_texts(
            [c["text"] for c in chunks],
            [{"id": c["id"]} for c in chunks],
        )
    return store.as_retriever(search_kwargs={"k": 5})

In [10]:
test_queries = []
for qid, qtext in queries.items():
    relevant_ids = list(qrels.get(qid, {}).keys())
    if not relevant_ids:
        continue  # skip queries with no relevance
    test_queries.append({
        "question": qtext,
        "relevant_doc_ids": relevant_ids
    })
print(len(test_queries))

323


In [11]:
def evaluate_retrieval(retriever, queries):
    recalls = []
    rranks = []

    for q in queries:
        retrieved_docs = retriever.invoke(q["question"])
        retrieved_ids = [d.metadata["id"] for d in retrieved_docs]

        relevant_ids = q["relevant_doc_ids"]
        recall = len(set(retrieved_ids) & set(relevant_ids)) / len(relevant_ids)
        recalls.append(recall)

        rank = next((i+1 for i, doc_id in enumerate(retrieved_ids) if doc_id in relevant_ids), None)
        rranks.append(1/rank if rank else 0)

    return {"recall@k": np.mean(recalls), "MRR": np.mean(rranks)}

In [12]:
for store_name in ["qdrant", "faiss", "chroma",]:
    retriever = build_vectorstore(store_name)
    metrics = evaluate_retrieval(retriever, test_queries)
    del retriever
    print(f"{store_name} retrieval metrics: {metrics}")


COLLECTIONS []
qdrant retrieval metrics: {'recall@k': np.float64(0.1076324185376777), 'MRR': np.float64(0.46274509803921576)}
faiss retrieval metrics: {'recall@k': np.float64(0.1076324185376777), 'MRR': np.float64(0.46480908152734784)}
chroma retrieval metrics: {'recall@k': np.float64(0.09806313930243787), 'MRR': np.float64(0.45103199174406605)}
